# Encoder-Decoder Architecture & Encoding Strategies

Christopher La Valle

---

This notebook covers:
1. **Encoding Strategies** — character, word, subword (BPE, WordPiece, Unigram)
2. **Encoder-Decoder Architecture** — attention, cross-attention, seq2seq
3. **Practical usage** with HuggingFace `transformers`

In [ ]:
import re
import math
import json
from collections import Counter, defaultdict

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

---
## 1 — Encoding Strategies

Before a model can process text it must be converted to integers — a **vocabulary** mapping tokens → IDs.  
The granularity of that split is the central design decision:

| Strategy | Unit | Vocab size | OOV handling |
|---|---|---|---|
| Character | single char | ~100–300 | none |
| Word | whitespace token | 50k–500k | `<UNK>` |
| Subword (BPE) | learned merge | 8k–100k | splits into known pieces |
| Subword (WordPiece) | learned merge (likelihood) | 8k–100k | splits into known pieces |
| Subword (Unigram) | probabilistic pruning | 8k–100k | splits into known pieces |

### 1.1 Character-Level Encoding

The simplest approach: each character is a distinct token drawn from an alphabet $\Sigma$.

**Formal definition.** Given an alphabet $\Sigma = \{c_1, \ldots, c_{|\Sigma|}\}$, define the vocabulary bijection

$$\phi : \Sigma \to \{0, 1, \ldots, |\Sigma|-1\}$$

A string $s = s_1 s_2 \cdots s_n \in \Sigma^*$ is encoded as the integer sequence $(\phi(s_1), \ldots, \phi(s_n))$.

**Information-theoretic lower bound.** By Shannon's source coding theorem, the minimum expected code length per character is the entropy of the character distribution:

$$H(\Sigma) = -\sum_{c \in \Sigma} p(c) \log_2 p(c) \quad \text{bits/character}$$

For English text $H \approx 1.3$ bits/char (Shannon, 1951), meaning a good model needs far fewer bits than the naïve $\log_2 |\Sigma| \approx 5$ bits.

**Sequence length.** A word of average length $\bar{\ell}$ produces $\bar{\ell}$ tokens, versus 1 token for word-level. For English, $\bar{\ell} \approx 5$, so character models process sequences roughly 5× longer — quadratic cost in Transformer self-attention.

**Pros:** $|\Sigma|$ is tiny (~100–300); zero out-of-vocabulary (OOV).  
**Cons:** very long sequences; model must learn morphology, orthography, and lexical semantics purely from co-occurrence patterns.

In [ ]:
class CharTokenizer:
    def __init__(self, corpus: str):
        chars = sorted(set(corpus))
        self.vocab = {c: i for i, c in enumerate(chars)}
        self.inv_vocab = {i: c for c, i in self.vocab.items()}

    def encode(self, text: str) -> list[int]:
        return [self.vocab[c] for c in text if c in self.vocab]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.inv_vocab[i] for i in ids)


corpus = "the quick brown fox jumps over the lazy dog"
char_tok = CharTokenizer(corpus)
sample = "the fox"
ids = char_tok.encode(sample)
print(f"Vocab size : {len(char_tok.vocab)}")
print(f"Tokens     : {list(sample)}")
print(f"IDs        : {ids}")
print(f"Decoded    : {char_tok.decode(ids)}")

### 1.2 Word-Level Encoding

Tokenise on whitespace (and punctuation), mapping each type to a unique integer.

**Zipf's law.** Word frequencies in natural language follow a power law:

$$f(r) \propto r^{-\alpha}, \quad \alpha \approx 1$$

where $r$ is the frequency rank. Consequently, vocabulary coverage grows as:

$$C(V) = \sum_{r=1}^{V} p(r) \approx 1 - \frac{\log(N/V)}{\log N}$$

Doubling vocabulary size yields diminishing coverage returns — the long tail of rare words is enormous.

**OOV probability.** For a corpus of $N$ tokens with vocabulary $V$ drawn from a Zipfian distribution, the probability that an unseen token from a new document is OOV is approximately:

$$P(\text{OOV}) \approx \left(\frac{N_{\text{new}}}{N_{\text{train}}}\right)^{1-\alpha}$$

This is non-negligible even for large training corpora.

**Vocabulary explosion.** Morphologically rich languages (Finnish, Turkish, Arabic) compound and inflect words extensively: a 500k-word German corpus might require a 500k+ vocabulary to cover 95% of types, whereas English needs ~50k.

**Pros:** each token carries full semantic meaning; sequences are short.  
**Cons:** vocabulary size is unbounded; morphological variants are unrelated embedding vectors; OOV collapses all unknown words to a single `<UNK>` representation.

In [ ]:
class WordTokenizer:
    UNK = "<UNK>"

    def __init__(self, corpus: str, min_freq: int = 1):
        counts = Counter(re.findall(r"\w+", corpus.lower()))
        words = [w for w, c in counts.items() if c >= min_freq]
        self.vocab = {self.UNK: 0, **{w: i + 1 for i, w in enumerate(words)}}
        self.inv_vocab = {i: w for w, i in self.vocab.items()}

    def encode(self, text: str) -> list[int]:
        return [self.vocab.get(w, 0) for w in re.findall(r"\w+", text.lower())]

    def decode(self, ids: list[int]) -> str:
        return " ".join(self.inv_vocab.get(i, self.UNK) for i in ids)


word_tok = WordTokenizer(corpus)
sample = "the quick fox swims"
ids = word_tok.encode(sample)
print(f"Vocab size : {len(word_tok.vocab)}")
print(f"IDs        : {ids}")
print(f"Decoded    : {word_tok.decode(ids)}  ← 'swims' is OOV → UNK")

---
### 1.3 Byte Pair Encoding (BPE)

BPE (Sennrich et al., 2016) adapts a lossless data compression algorithm to vocabulary induction.

#### Formal Setup

Represent the corpus as a multiset of (word-type, frequency) pairs:

$$\mathcal{D} = \{(w_1, f_1), (w_2, f_2), \ldots, (w_T, f_T)\}, \quad w_i \in \Sigma^+, \; f_i \in \mathbb{Z}_{>0}$$

Each word is initially split into its character sequence plus an end-of-word marker:

$$w = c_1 c_2 \cdots c_k \;\Rightarrow\; (c_1, c_2, \ldots, c_k, \texttt{</w>})$$

The initial vocabulary is $\mathcal{V}_0 = \Sigma \cup \{\texttt{</w>}\}$.

#### Pair Statistics

At step $t$, define the pair frequency function:

$$\text{freq}_t(a, b) = \sum_{(w, f) \in \mathcal{D}_t} f \cdot \#(a, b \;\text{adjacent in}\; w)$$

where $\#(a,b \text{ adjacent in } w)$ counts non-overlapping occurrences of the bigram $(a, b)$ in the current segmentation of $w$.

#### Merge Rule

Select the most frequent pair:

$$(\hat{a}, \hat{b}) = \arg\max_{(a,b)} \text{freq}_t(a, b)$$

Replace every occurrence of $(\hat{a}, \hat{b})$ with the new symbol $\hat{a}\hat{b}$, and add $\hat{a}\hat{b}$ to the vocabulary:

$$\mathcal{V}_{t+1} = \mathcal{V}_t \cup \{\hat{a}\hat{b}\}$$

Repeat for $K$ steps, yielding a final vocabulary $\mathcal{V}_K$ of size $|\mathcal{V}_0| + K$.

#### Compression Interpretation

Each merge step reduces the total number of tokens in the corpus by exactly $\text{freq}_t(\hat{a}, \hat{b})$. BPE therefore greedily minimises description length — it is a greedy approximation to Minimum Description Length (MDL) tokenisation.

#### Encoding at Inference

Given learned merges $m_1, m_2, \ldots, m_K$ (in training order), encode a new word by:
1. Split into characters.
2. Apply each merge rule $m_i = (a_i, b_i)$ left-to-right, greedily merging the first matching adjacent pair.

**Time complexity:** Training is $O(K \cdot N)$ where $N$ is corpus size. Encoding a word of length $\ell$ is $O(K \cdot \ell)$.

**Determinism:** BPE encoding is deterministic — unlike the Unigram model, there is no ambiguity.

In [ ]:
def get_vocab(corpus: str) -> dict[tuple, int]:
    """Split each word into characters + end-of-word marker."""
    vocab: dict[tuple, int] = {}
    for word in corpus.split():
        chars = tuple(list(word) + ["</w>"])
        vocab[chars] = vocab.get(chars, 0) + 1
    return vocab


def get_pair_stats(vocab: dict[tuple, int]) -> Counter:
    pairs: Counter = Counter()
    for word, freq in vocab.items():
        for a, b in zip(word, word[1:]):
            pairs[(a, b)] += freq
    return pairs


def merge_pair(pair: tuple, vocab: dict[tuple, int]) -> dict[tuple, int]:
    new_vocab: dict[tuple, int] = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        new_word = " ".join(word)
        new_word = new_word.replace(bigram, replacement)
        new_vocab[tuple(new_word.split())] = freq
    return new_vocab


def train_bpe(corpus: str, num_merges: int) -> list[tuple]:
    vocab = get_vocab(corpus)
    merges: list[tuple] = []
    for _ in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        vocab = merge_pair(best, vocab)
        merges.append(best)
    return merges, vocab


train_corpus = "low low low low lower lower newest newest newest widest widest"
merges, final_vocab = train_bpe(train_corpus, num_merges=10)

print("Learned BPE merges (in order):")
for i, m in enumerate(merges, 1):
    print(f"  {i:2d}. {m[0] + ' ' + m[1]:20s} → {''.join(m)}")

print("\nFinal vocabulary:")
for token in sorted({t for w in final_vocab for t in w}):
    print(f"  {token}")

In [ ]:
def bpe_encode(word: str, merges: list[tuple]) -> list[str]:
    """Apply learned BPE merges to a single word."""
    tokens = list(word) + ["</w>"]
    for pair in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


for word in ["low", "lower", "newest", "widest", "lowest"]:
    toks = bpe_encode(word, merges)
    print(f"  {word:10s} → {toks}")

In [ ]:
# Visualise vocab growth as merges accumulate
def vocab_size_over_merges(corpus: str, max_merges: int) -> list[int]:
    base_vocab = get_vocab(corpus)
    sizes = [len({t for w in base_vocab for t in w})]
    for k in range(1, max_merges + 1):
        _, voc = train_bpe(corpus, k)
        sizes.append(len({t for w in voc for t in w}))
    return sizes


sizes = vocab_size_over_merges(train_corpus, 10)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(sizes))), y=sizes, mode="lines+markers",
                         marker=dict(size=8), line=dict(width=2)))
fig.update_layout(title="BPE: Vocabulary Size vs. Number of Merges",
                  xaxis_title="Merge step", yaxis_title="Vocabulary size",
                  template="plotly_dark", height=380)
fig.show()

---
### 1.4 WordPiece

Used by **BERT**, **DistilBERT**, and **ELECTRA** (Schuster & Nakamura, 2012; Devlin et al., 2019).

#### Likelihood Objective

Instead of frequency, WordPiece selects the merge that maximises the **log-likelihood** of the corpus under a unigram language model fitted to the current vocabulary.

Given current vocabulary $\mathcal{V}$, the unigram LM assigns each token $t \in \mathcal{V}$ probability:

$$p(t) = \frac{\text{count}(t)}{\sum_{t' \in \mathcal{V}} \text{count}(t')}$$

The corpus log-likelihood is:

$$\mathcal{L}(\mathcal{V}) = \sum_{(w, f) \in \mathcal{D}} f \cdot \log P^*(w), \quad P^*(w) = \prod_{t \in \text{seg}^*(w)} p(t)$$

where $\text{seg}^*(w)$ is the maximum-probability segmentation of $w$.

#### Merge Scoring via PMI

Merging the pair $(A, B)$ increases the likelihood by:

$$\Delta\mathcal{L}(A, B) \approx \text{freq}(AB) \cdot \log \frac{p(AB)}{p(A)\,p(B)}$$

The score used in practice is proportional to the **Pointwise Mutual Information (PMI)**:

$$\text{score}(A, B) = \frac{\text{freq}(AB)}{\text{freq}(A) \cdot \text{freq}(B)}$$

which equals $\frac{p(AB)}{p(A)\,p(B)}$ up to a constant. PMI measures how much more often $A$ and $B$ co-occur adjacent than expected under independence.

**Key difference from BPE:** BPE is a pure frequency counter; WordPiece normalises by the individual token frequencies, so it resists merging tokens that are separately very common (e.g., `t` and `he` → `the` would be penalised because both `t` and `he` are frequent).

#### Subword Prefix Convention

WordPiece prefixes continuation tokens with `##`:

$$\text{"playing"} \;\to\; [\texttt{play},\; \texttt{\#\#ing}]$$

This allows the model to reconstruct the original word boundary from the token sequence.

In [ ]:
def train_wordpiece(corpus: str, num_merges: int) -> list[tuple]:
    vocab = get_vocab(corpus)
    merges: list[tuple] = []

    for _ in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break

        # Unigram frequencies
        unigram: Counter = Counter()
        for word, freq in vocab.items():
            for tok in word:
                unigram[tok] += freq
        total = sum(unigram.values())

        best, best_score = None, -1
        for (a, b), freq in pairs.items():
            score = freq / (unigram[a] * unigram[b] / total)
            if score > best_score:
                best_score, best = score, (a, b)

        vocab = merge_pair(best, vocab)
        merges.append(best)

    return merges


wp_merges = train_wordpiece(train_corpus, num_merges=10)
print("WordPiece merges:")
for i, m in enumerate(wp_merges, 1):
    print(f"  {i:2d}. {m[0] + ' ' + m[1]:20s} → {''.join(m)}")

---
### 1.5 Unigram Language Model (SentencePiece)

Used by **T5**, **XLNet**, **mBART**, **ALBERT**, and **LLaMA** via SentencePiece (Kudo & Richardson, 2018).

#### Model Definition

A unigram LM defines a probability over segmentations. Let $\mathcal{V}$ be the current vocabulary. For a word $w$, let $S(w)$ be the set of all valid segmentations into tokens from $\mathcal{V}$. The probability of segmentation $\mathbf{x} = (x_1, \ldots, x_M) \in S(w)$ is:

$$P(\mathbf{x}) = \prod_{i=1}^{M} p(x_i), \quad \sum_{t \in \mathcal{V}} p(t) = 1, \quad p(t) > 0 \;\forall\, t$$

The marginal probability of word $w$ is:

$$P(w) = \sum_{\mathbf{x} \in S(w)} P(\mathbf{x})$$

#### EM Training

The corpus log-likelihood is:

$$\mathcal{L} = \sum_{(w, f) \in \mathcal{D}} f \cdot \log P(w)$$

This is maximised via **Expectation-Maximisation**:

**E-step:** Compute the expected count of each token $t$ using the forward-backward algorithm over the lattice of segmentations:

$$\mathbb{E}[\text{count}(t)] = \sum_{(w,f) \in \mathcal{D}} f \cdot \sum_{\mathbf{x} \in S(w)} \frac{P(\mathbf{x})}{P(w)} \cdot \#(t \in \mathbf{x})$$

In practice the sum over all segmentations is computed efficiently with dynamic programming on the segmentation lattice (complexity $O(\ell^2)$ per word).

**M-step:** Re-estimate token probabilities by normalised expected counts:

$$p(t) \leftarrow \frac{\mathbb{E}[\text{count}(t)]}{\sum_{t'} \mathbb{E}[\text{count}(t')]}$$

#### Vocabulary Pruning

After EM convergence, compute the **loss increase** if token $t$ were removed:

$$\Delta\mathcal{L}(t) = \mathcal{L}_{\mathcal{V}} - \mathcal{L}_{\mathcal{V} \setminus \{t\}}$$

Tokens with the smallest $\Delta\mathcal{L}$ (removing them hurts least) are pruned. Characters are always kept to guarantee full coverage. Prune a fixed fraction (e.g., 20%) and repeat until target vocabulary size is reached.

#### Viterbi Decoding

The best segmentation of word $w$ is found by Viterbi dynamic programming:

$$\delta_j = \max_{i < j,\, w[i:j] \in \mathcal{V}} \bigl(\delta_i + \log p(w[i:j])\bigr), \quad \delta_0 = 0$$

This runs in $O(\ell^2)$ time for a word of length $\ell$.

#### Stochastic Segmentation

During training the Unigram model can **sample** segmentations proportional to $P(\mathbf{x})$ rather than always taking the argmax. This acts as a data augmentation / regulariser, making the model more robust to segmentation ambiguity.

In [ ]:
def build_unigram_model(corpus: str, vocab_size: int = 20) -> dict[str, float]:
    """Simplified Unigram LM: use substring frequency as a proxy for log-prob."""
    words = corpus.split()
    substr_count: Counter = Counter()
    for word in words:
        w = word + "</w>"
        for start in range(len(w)):
            for end in range(start + 1, len(w) + 1):
                substr_count[w[start:end]] += 1

    # Keep the top-k substrings by count (ensure all chars are included)
    chars = {c for word in words for c in word} | {"</w>"}
    candidates = {s: c for s, c in substr_count.items() if len(s) > 1}
    top = sorted(candidates, key=candidates.get, reverse=True)[: vocab_size - len(chars)]
    vocab = list(chars) + top

    total = sum(substr_count[t] for t in vocab)
    probs = {t: substr_count[t] / total for t in vocab}
    return probs


def viterbi_segment(word: str, probs: dict[str, float]) -> list[str]:
    """Viterbi decoding: find highest-probability segmentation."""
    w = word + "</w>"
    n = len(w)
    best = [(-math.inf, [])] * (n + 1)
    best[0] = (0.0, [])
    for i in range(n):
        if best[i][0] == -math.inf:
            continue
        for j in range(i + 1, n + 1):
            tok = w[i:j]
            if tok in probs:
                score = best[i][0] + math.log(probs[tok])
                if score > best[j][0]:
                    best[j] = (score, best[i][1] + [tok])
    return best[n][1]


model = build_unigram_model(train_corpus, vocab_size=25)
print(f"Unigram vocab size: {len(model)}")
for word in ["low", "lower", "newest", "widest"]:
    seg = viterbi_segment(word, model)
    print(f"  {word:10s} → {seg}")

---
### 1.6 Comparing Strategies Side-by-Side

We compare how each strategy tokenizes the same sentence and the resulting sequence lengths.

In [ ]:
sentences = [
    "the quick brown fox jumps over the lazy dog",
    "tokenization is the lowest level of text preprocessing",
    "newer wider lower",
]

full_corpus = " ".join(sentences)
c_tok = CharTokenizer(full_corpus)
w_tok = WordTokenizer(full_corpus)
bpe_merges_full, _ = train_bpe(full_corpus, num_merges=30)
uni_model = build_unigram_model(full_corpus, vocab_size=60)

rows = []
for sent in sentences:
    char_toks = list(sent)
    word_toks = re.findall(r"\w+", sent.lower())
    bpe_toks = [t for w in word_toks for t in bpe_encode(w, bpe_merges_full)]
    uni_toks = [t for w in word_toks for t in viterbi_segment(w, uni_model)]
    rows.append({"sentence": sent[:40], "char": len(char_toks),
                 "word": len(word_toks), "bpe": len(bpe_toks), "unigram": len(uni_toks)})

strategies = ["char", "word", "bpe", "unigram"]
colors = px.colors.qualitative.Plotly
fig = go.Figure()
for i, s in enumerate(strategies):
    fig.add_trace(go.Bar(
        name=s,
        x=[r["sentence"] for r in rows],
        y=[r[s] for r in rows],
        marker_color=colors[i],
    ))
fig.update_layout(barmode="group", title="Token Count per Strategy",
                  xaxis_title="Sentence", yaxis_title="# tokens",
                  template="plotly_dark", height=420)
fig.show()

### 1.7 HuggingFace Tokenizers in Practice

In [ ]:
from transformers import AutoTokenizer

# GPT-2 uses BPE; BERT uses WordPiece; T5 uses Unigram/SentencePiece
model_names = {
    "BPE (GPT-2)": "gpt2",
    "WordPiece (BERT)": "bert-base-uncased",
    "Unigram (T5)": "t5-small",
}

test_sentence = "The tokenization of uncommon words like 'cryoturbation' varies."

for label, name in model_names.items():
    tok = AutoTokenizer.from_pretrained(name)
    tokens = tok.tokenize(test_sentence)
    print(f"\n{label}")
    print(f"  vocab size : {tok.vocab_size:,}")
    print(f"  tokens     : {tokens}")
    print(f"  length     : {len(tokens)}")

---
## 2 — Encoder-Decoder Architecture

The Transformer encoder-decoder (Vaswani et al., 2017) defines a conditional distribution over target sequences $Y$ given source sequences $X$:

$$P(Y \mid X) = \prod_{t=1}^{|Y|} P(y_t \mid y_{<t}, X)$$

The encoder maps $X$ to a sequence of context vectors (the **memory**), and the decoder factorises the target distribution autoregressively conditioned on that memory.

```
Source tokens                          Target tokens (shifted right)
     │                                        │
  Embedding + PE                        Embedding + PE
     │                                        │
 ┌───┴──────────┐                      ┌──────┴──────────────────┐
 │   ENCODER    │  context vectors →   │       DECODER           │
 │  Self-Attn   │                      │  Masked Self-Attn       │
 │  FFN × N     │                      │  Cross-Attn (← encoder) │
 └──────────────┘                      │  FFN × M                │
                                       └─────────────────────────┘
                                                  │
                                             Linear + Softmax
                                                  │
                                           Next-token logits
```

**Training objective.** The model is trained to minimise the cross-entropy loss (equivalently, maximise the log-likelihood) over a parallel corpus $\{(X^{(n)}, Y^{(n)})\}$:

$$\mathcal{L} = -\frac{1}{N} \sum_{n=1}^N \sum_{t=1}^{|Y^{(n)}|} \log P(y_t^{(n)} \mid y_{<t}^{(n)}, X^{(n)})$$

**Encoder:** bidirectional — all positions attend to all positions, building rich contextual representations.  
**Decoder:** autoregressive — causal masking enforces that position $t$ only attends to positions $< t$ in the target, preserving the autoregressive factorisation.

### 2.1 Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

where $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, $V \in \mathbb{R}^{m \times d_v}$.

#### Interpretation as Soft Dictionary Lookup

Attention can be read as a **differentiable dictionary**:
- **Keys** $K$ index entries in the dictionary.
- **Values** $V$ hold the associated data.
- **Queries** $Q$ retrieve a weighted sum of values, where weights reflect similarity to keys.

The similarity metric is the scaled inner product $\frac{q_i \cdot k_j}{\sqrt{d_k}}$, which measures alignment between query $i$ and key $j$.

#### Why $\sqrt{d_k}$ Scaling?

If $q, k \sim \mathcal{N}(0, 1)$ independently, then $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has:

$$\mathbb{E}[q \cdot k] = 0, \qquad \text{Var}[q \cdot k] = d_k$$

so $\text{std}[q \cdot k] = \sqrt{d_k}$. Without scaling, dot products grow proportionally to $\sqrt{d_k}$, pushing the softmax argument into regions of extreme values. In those regions:

$$\frac{\partial}{\partial z_i}\text{softmax}(z)_i = \text{softmax}(z)_i (1 - \text{softmax}(z)_i) \approx 0$$

causing **vanishing gradients**. Dividing by $\sqrt{d_k}$ restores unit variance regardless of $d_k$.

#### Computational Complexity

| Operation | Time | Space |
|---|---|---|
| $QK^\top$ | $O(n m d_k)$ | $O(nm)$ |
| Softmax | $O(nm)$ | $O(nm)$ |
| $\text{weights} \cdot V$ | $O(n m d_v)$ | $O(n d_v)$ |

For self-attention ($n = m$), the dominant cost is $O(n^2 d)$ time and $O(n^2)$ space — the quadratic bottleneck that motivates sparse/linear attention variants.

#### Softmax Temperature

The sharpness of the attention distribution is controlled by temperature $\tau$:

$$\text{softmax}\!\left(\frac{QK^\top}{\tau \sqrt{d_k}}\right)$$

- $\tau \to 0$: hard argmax (attend to a single key)
- $\tau \to \infty$: uniform distribution over all keys (no attention)
- $\tau = 1$: standard scaled attention

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray]:
    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    weights = softmax(scores)
    return weights @ V, weights


# Tiny demo: 4 query positions, 6 key/value positions, d_k = 8
rng = np.random.default_rng(42)
Q = rng.normal(size=(4, 8))
K = rng.normal(size=(6, 8))
V = rng.normal(size=(6, 8))

output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights (4 queries × 6 keys):")
print(np.round(weights, 3))

fig = px.imshow(weights, color_continuous_scale="Blues",
                labels=dict(x="Key position", y="Query position", color="Weight"),
                title="Attention Weight Heatmap")
fig.update_layout(template="plotly_dark", height=320)
fig.show()

In [ ]:
# Empirically verify the √d_k variance argument
d_ks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
n_samples = 10_000
rng = np.random.default_rng(0)

raw_stds, scaled_stds = [], []
for d_k in d_ks:
    q = rng.normal(size=(n_samples, d_k))
    k = rng.normal(size=(n_samples, d_k))
    dots = (q * k).sum(axis=1)           # raw dot products
    raw_stds.append(dots.std())
    scaled_stds.append((dots / math.sqrt(d_k)).std())

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Unscaled: std ≈ √d_k", "Scaled: std ≈ 1"])
fig.add_trace(go.Scatter(x=d_ks, y=raw_stds, mode="lines+markers", name="std(q·k)",
                         line=dict(color="tomato")), row=1, col=1)
fig.add_trace(go.Scatter(x=d_ks, y=[math.sqrt(d) for d in d_ks], mode="lines",
                         name="√d_k (theory)", line=dict(dash="dash", color="orange")),
              row=1, col=1)
fig.add_trace(go.Scatter(x=d_ks, y=scaled_stds, mode="lines+markers",
                         name="std(q·k / √d_k)", line=dict(color="steelblue")), row=1, col=2)
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=1, col=2)
fig.update_xaxes(type="log", title_text="d_k")
fig.update_yaxes(title_text="Standard deviation")
fig.update_layout(title="Why Attention Scales by √d_k", template="plotly_dark", height=380)
fig.show()

### 2.2 Multi-Head Attention

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O \in \mathbb{R}^{n \times d_{\text{model}}}$$

$$\text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

with projection matrices $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$, $W^O \in \mathbb{R}^{h d_v \times d_{\text{model}}}$.

The standard choice is $d_k = d_v = d_{\text{model}} / h$.

#### Parameter Count

| Matrix | Shape | Parameters |
|---|---|---|
| $W_i^Q$ (per head) | $d_{\text{model}} \times d_k$ | $d_{\text{model}}^2 / h$ |
| $W_i^K$ (per head) | $d_{\text{model}} \times d_k$ | $d_{\text{model}}^2 / h$ |
| $W_i^V$ (per head) | $d_{\text{model}} \times d_v$ | $d_{\text{model}}^2 / h$ |
| $W^O$ | $h d_v \times d_{\text{model}}$ | $d_{\text{model}}^2$ |
| **Total** | | $4 d_{\text{model}}^2$ |

The total parameter count is independent of $h$ — more heads partition the same budget into more specialised subspaces.

#### Subspace Interpretation

Each head $i$ projects into a $d_k$-dimensional subspace and computes attention there. Different heads can specialise in different types of relationships:
- **Syntactic heads** — attend to dependency relations (subject → verb)
- **Positional heads** — attend to adjacent positions
- **Coreference heads** — link pronouns to their antecedents
- **Rare-token heads** — attend to low-frequency tokens in the context

This was empirically demonstrated by Clark et al. (2019) via probing classifiers on BERT attention heads.

#### Grouped-Query Attention (GQA)

Modern LLMs (LLaMA-2, Mistral) use **Grouped-Query Attention** to reduce the KV cache:

$$\text{GQA}: h_Q \text{ query heads}, \; h_{KV} \text{ key/value heads}, \; h_{KV} \ll h_Q$$

Each group of $h_Q / h_{KV}$ query heads shares a single K/V projection. Memory scales with $h_{KV}$, not $h_Q$.

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int, rng=None):
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads
        rng = rng or np.random.default_rng(0)
        scale = 0.1
        self.Wq = rng.normal(scale=scale, size=(d_model, d_model))
        self.Wk = rng.normal(scale=scale, size=(d_model, d_model))
        self.Wv = rng.normal(scale=scale, size=(d_model, d_model))
        self.Wo = rng.normal(scale=scale, size=(d_model, d_model))

    def _split_heads(self, x: np.ndarray) -> np.ndarray:
        # (seq, d_model) → (h, seq, d_k)
        seq = x.shape[0]
        return x.reshape(seq, self.h, self.d_k).transpose(1, 0, 2)

    def forward(self, Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                mask: np.ndarray | None = None) -> np.ndarray:
        Qs = self._split_heads(Q @ self.Wq)
        Ks = self._split_heads(K @ self.Wk)
        Vs = self._split_heads(V @ self.Wv)

        heads = []
        for i in range(self.h):
            out, _ = scaled_dot_product_attention(Qs[i], Ks[i], Vs[i], mask)
            heads.append(out)

        concat = np.concatenate(heads, axis=-1)  # (seq, d_model)
        return concat @ self.Wo


d_model, n_heads, seq_len = 32, 4, 6
mha = MultiHeadAttention(d_model, n_heads, rng=np.random.default_rng(7))
x = np.random.default_rng(1).normal(size=(seq_len, d_model))
out = mha.forward(x, x, x)  # self-attention
print(f"Input  shape: {x.shape}")
print(f"Output shape: {out.shape}")

### 2.3 Positional Encoding

Attention is **permutation-equivariant**: $\text{MHA}(PX, PX, PX) = P\,\text{MHA}(X,X,X)$ for any permutation matrix $P$. Position information must be injected explicitly.

#### Sinusoidal Encoding (Vaswani et al., 2017)

$$PE_{(\text{pos}, 2i)} = \sin\!\left(\frac{\text{pos}}{10000^{2i/d}}\right), \quad
PE_{(\text{pos}, 2i+1)} = \cos\!\left(\frac{\text{pos}}{10000^{2i/d}}\right)$$

**Relative position property.** For any fixed offset $k$, $PE_{\text{pos}+k}$ is a **linear function** of $PE_{\text{pos}}$:

$$\begin{pmatrix} \sin((\text{pos}+k)\,\omega) \\ \cos((\text{pos}+k)\,\omega) \end{pmatrix} = \underbrace{\begin{pmatrix} \cos(k\omega) & \sin(k\omega) \\ -\sin(k\omega) & \cos(k\omega) \end{pmatrix}}_{R_k} \begin{pmatrix} \sin(\text{pos}\,\omega) \\ \cos(\text{pos}\,\omega) \end{pmatrix}$$

where $\omega = 1/10000^{2i/d}$. This rotation matrix $R_k$ is the same for all positions, so the model can potentially learn to attend by relative offset.

#### Rotary Positional Embeddings (RoPE)

RoPE (Su et al., 2021), used by LLaMA, GPT-NeoX, applies rotations **directly to the query and key vectors** so that the inner product $q_m^\top k_n$ depends only on the relative offset $m - n$:

$$\langle f(q, m),\; f(k, n) \rangle = g(q, k, m-n)$$

Concretely, for each pair of dimensions $(2i, 2i+1)$ at position $\text{pos}$:

$$\begin{pmatrix} q_{2i}' \\ q_{2i+1}' \end{pmatrix} = \begin{pmatrix} \cos(\text{pos}\,\theta_i) & -\sin(\text{pos}\,\theta_i) \\ \sin(\text{pos}\,\theta_i) & \phantom{-}\cos(\text{pos}\,\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}, \quad \theta_i = 10000^{-2i/d}$$

RoPE generalises to longer sequences than seen during training via **NTK-aware scaling**: $\theta_i \to \theta_i / \lambda^{2i/d}$ for scale factor $\lambda$.

#### ALiBi (Press et al., 2022)

Rather than adding positional encodings to embeddings, ALiBi subtracts a **linear bias** from attention logits:

$$\text{score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} - m_h \cdot |i - j|$$

where slope $m_h$ is a fixed, head-specific constant. This imposes a recency inductive bias and extrapolates well beyond training length.

In [ ]:
def sinusoidal_pe(max_len: int, d_model: int) -> np.ndarray:
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-math.log(10000) / d_model))
    PE[:, 0::2] = np.sin(pos * div)
    PE[:, 1::2] = np.cos(pos * div)
    return PE


pe = sinusoidal_pe(max_len=50, d_model=64)
fig = px.imshow(pe.T, aspect="auto", color_continuous_scale="RdBu",
                labels=dict(x="Position", y="Dimension", color="Value"),
                title="Sinusoidal Positional Encodings")
fig.update_layout(template="plotly_dark", height=380)
fig.show()

In [ ]:
# RoPE implementation and comparison with sinusoidal
def rope_rotate(x: np.ndarray, pos: int, theta_base: float = 10000.0) -> np.ndarray:
    """Apply RoPE rotation to a vector x at position pos."""
    d = x.shape[-1]
    assert d % 2 == 0
    out = x.copy()
    for i in range(d // 2):
        theta = pos / (theta_base ** (2 * i / d))
        cos_t, sin_t = math.cos(theta), math.sin(theta)
        x0, x1 = x[2 * i], x[2 * i + 1]
        out[2 * i]     =  cos_t * x0 - sin_t * x1
        out[2 * i + 1] =  sin_t * x0 + cos_t * x1
    return out


# Show that RoPE inner product depends only on relative position m-n
rng = np.random.default_rng(3)
d = 16
q = rng.normal(size=d)
k = rng.normal(size=d)

positions = range(20)
# Fix query at position 10, vary key position
q_rot = rope_rotate(q, pos=10)
scores = [np.dot(q_rot, rope_rotate(k, pos=p)) for p in positions]

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(positions), y=scores, mode="lines+markers",
                         name="RoPE score (query fixed at pos=10)"))
fig.add_vline(x=10, line_dash="dash", line_color="gray", annotation_text="query pos")
fig.update_layout(title="RoPE: Inner Product as a Function of Key Position\n"
                        "(score depends on relative offset, not absolute position)",
                  xaxis_title="Key position", yaxis_title="q_rot · k_rot",
                  template="plotly_dark", height=360)
fig.show()

# Verify: score(q@10, k@13) == score(q@5, k@8) — same relative offset of 3
score_10_13 = np.dot(rope_rotate(q, 10), rope_rotate(k, 13))
score_5_8   = np.dot(rope_rotate(q, 5),  rope_rotate(k, 8))
print(f"score(pos=10, pos=13): {score_10_13:.6f}")
print(f"score(pos= 5, pos= 8): {score_5_8:.6f}  ← same relative offset → same score")

### 2.4 Encoder Block

Each encoder layer $\ell$ applies two sub-layers with residual connections (He et al., 2016) and Layer Normalisation (Ba et al., 2016).

#### Post-LN (original Transformer)

$$\mathbf{a}^\ell = \text{LayerNorm}\!\left(\mathbf{x}^{\ell-1} + \text{MHA}(\mathbf{x}^{\ell-1})\right)$$
$$\mathbf{x}^\ell = \text{LayerNorm}\!\left(\mathbf{a}^\ell + \text{FFN}(\mathbf{a}^\ell)\right)$$

#### Pre-LN (GPT-2, LLaMA, most modern models)

$$\mathbf{a}^\ell = \mathbf{x}^{\ell-1} + \text{MHA}\!\left(\text{LayerNorm}(\mathbf{x}^{\ell-1})\right)$$
$$\mathbf{x}^\ell = \mathbf{a}^\ell + \text{FFN}\!\left(\text{LayerNorm}(\mathbf{a}^\ell)\right)$$

Pre-LN has smoother gradients and does not require learning rate warm-up, but can underperform Post-LN at final converged quality (Liu et al., 2020).

#### Layer Normalisation

$$\text{LayerNorm}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sigma + \epsilon} \odot \gamma + \beta, \quad \mu = \frac{1}{d}\sum_j x_j, \quad \sigma^2 = \frac{1}{d}\sum_j (x_j - \mu)^2$$

Learnable $\gamma, \beta \in \mathbb{R}^d$ restore the model's capacity to represent any mean and variance after normalisation. Unlike **Batch Normalisation**, Layer Norm normalises over the feature dimension rather than the batch dimension, making it sequence-length and batch-size agnostic.

**RMS Norm** (used by LLaMA): drops the mean-centering step, normalising by root mean square only:

$$\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\frac{1}{d}\sum_j x_j^2 + \epsilon}} \odot \gamma$$

#### Feed-Forward Network (FFN)

$$\text{FFN}(\mathbf{x}) = W_2 \,\sigma\!\left(W_1 \mathbf{x} + b_1\right) + b_2, \quad W_1 \in \mathbb{R}^{d_{\text{ff}} \times d}, \; W_2 \in \mathbb{R}^{d \times d_{\text{ff}}}$$

Applied **position-wise** (identically and independently to each token). The expansion ratio $d_{\text{ff}} / d = 4$ was set empirically in the original paper; the extra capacity acts as a key-value memory (Geva et al., 2021).

**GELU** activation (Hendrycks & Gimpel, 2016), used by GPT and BERT:

$$\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5x\!\left(1 + \tanh\!\left[\sqrt{\tfrac{2}{\pi}}\left(x + 0.044715x^3\right)\right]\right)$$

where $\Phi$ is the standard Gaussian CDF. GELU is a smooth approximation to ReLU that weights inputs by their probability under a standard normal — inputs unlikely to be noise pass through unchanged.

**SwiGLU** (Shazeer, 2020), used by LLaMA, PaLM:

$$\text{SwiGLU}(\mathbf{x}) = \left(\text{Swish}(W_1 \mathbf{x})\right) \odot \left(W_3 \mathbf{x}\right), \quad \text{Swish}(x) = x \cdot \sigma(x)$$

Introduces a learned gating mechanism; typically requires shrinking $d_{\text{ff}}$ to keep the parameter count equal to the standard FFN.

#### Total Encoder Parameter Count (per layer)

| Component | Parameters |
|---|---|
| MHA ($W^Q, W^K, W^V, W^O$) | $4\,d^2$ |
| LayerNorm ×2 ($\gamma, \beta$) | $4d$ |
| FFN ($W_1, b_1, W_2, b_2$) | $2\,d\,d_{\text{ff}} + d_{\text{ff}} + d$ |
| **Total per layer** | $\approx 4d^2 + 8d^2 = 12\,d^2$ (with $d_{\text{ff}}=4d$) |

In [ ]:
def layer_norm(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))


class FFN:
    def __init__(self, d_model: int, d_ff: int, rng):
        self.W1 = rng.normal(scale=0.1, size=(d_model, d_ff))
        self.W2 = rng.normal(scale=0.1, size=(d_ff, d_model))

    def forward(self, x: np.ndarray) -> np.ndarray:
        return gelu(x @ self.W1) @ self.W2


class EncoderLayer:
    def __init__(self, d_model: int, num_heads: int, d_ff: int, rng):
        self.mha = MultiHeadAttention(d_model, num_heads, rng)
        self.ffn = FFN(d_model, d_ff, rng)

    def forward(self, x: np.ndarray) -> np.ndarray:
        x = layer_norm(x + self.mha.forward(x, x, x))  # self-attn + residual
        x = layer_norm(x + self.ffn.forward(x))         # ffn + residual
        return x


rng = np.random.default_rng(99)
enc_layer = EncoderLayer(d_model=32, num_heads=4, d_ff=64, rng=rng)
src = rng.normal(size=(8, 32))  # 8 tokens, d_model=32
enc_out = enc_layer.forward(src)
print(f"Encoder layer output: {enc_out.shape}  (seq_len=8, d_model=32)")

### 2.5 Decoder Block

The decoder factorises the target distribution autoregressively:

$$P(Y \mid X) = \prod_{t=1}^{T} P(y_t \mid y_1, \ldots, y_{t-1}, X)$$

This requires **causal masking** so that position $t$ cannot attend to positions $> t$ during training.

#### Causal Mask

Define the additive mask $M \in \{0, -\infty\}^{T \times T}$:

$$M_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

The masked attention score becomes:

$$\text{score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} + M_{ij}$$

After softmax, positions with $M_{ij} = -\infty$ receive weight $e^{-\infty} = 0$, effectively preventing information flow from future tokens.

#### Three Sub-Layers (Post-LN formulation)

$$\mathbf{s}^\ell = \text{LayerNorm}\!\left(\mathbf{x}^{\ell-1} + \text{MaskedMHA}(\mathbf{x}^{\ell-1})\right)$$
$$\mathbf{a}^\ell = \text{LayerNorm}\!\left(\mathbf{s}^\ell + \text{CrossMHA}(\mathbf{s}^\ell,\; \text{memory},\; \text{memory})\right)$$
$$\mathbf{x}^\ell = \text{LayerNorm}\!\left(\mathbf{a}^\ell + \text{FFN}(\mathbf{a}^\ell)\right)$$

where $\text{memory} = \text{Encoder}(X) \in \mathbb{R}^{|X| \times d}$ is fixed across all decoder layers.

In the cross-attention sublayer:
- **Queries** come from the decoder state $\mathbf{s}^\ell$ (what the decoder is currently predicting)
- **Keys and Values** come from the encoder memory (what information is available from the source)

#### KV Cache at Inference

During autoregressive generation, each step $t$ extends the target sequence by one token. Naïvely recomputing all attention from scratch would cost $O(t^2 d)$ per step. The **KV cache** stores the key and value projections for all previous positions:

$$\text{Cache}^{\ell} = \{(K_i^\ell, V_i^\ell)\}_{i=1}^{t-1}$$

At step $t$, only the new query $q_t$ is computed; it attends to the cached keys and values in $O(t\,d)$ time. Total generation cost is $O(T^2 d)$, but with no redundant computation.

**Memory:** The KV cache occupies $2 \times N_{\text{layers}} \times h \times d_k \times T \times \text{batch}$ elements — the dominant memory cost during inference for large models.

#### Decoder-Only vs Encoder-Decoder

| | Encoder-Decoder (T5) | Decoder-Only (GPT) |
|---|---|---|
| Architecture | 2 stacks | 1 stack |
| Cross-attention | yes | no |
| Conditioning | explicit (encoder) | in-context (prepended) |
| Best for | seq2seq tasks | open-ended generation |
| Efficiency | source encoded once | full sequence on every step |

In [ ]:
def causal_mask(seq_len: int) -> np.ndarray:
    """Upper-triangular True = positions to mask (future)."""
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)


class DecoderLayer:
    def __init__(self, d_model: int, num_heads: int, d_ff: int, rng):
        self.self_mha  = MultiHeadAttention(d_model, num_heads, rng)
        self.cross_mha = MultiHeadAttention(d_model, num_heads, rng)
        self.ffn       = FFN(d_model, d_ff, rng)

    def forward(self, tgt: np.ndarray, memory: np.ndarray) -> np.ndarray:
        tgt_len = tgt.shape[0]
        mask = causal_mask(tgt_len)
        tgt = layer_norm(tgt + self.self_mha.forward(tgt, tgt, tgt, mask))   # masked self-attn
        tgt = layer_norm(tgt + self.cross_mha.forward(tgt, memory, memory))  # cross-attn
        tgt = layer_norm(tgt + self.ffn.forward(tgt))
        return tgt


# Visualise causal mask
mask = causal_mask(6).astype(float)
fig = px.imshow(mask, color_continuous_scale="Greys",
                labels=dict(x="Key (source)", y="Query (target)"),
                title="Causal Mask (masked positions = 1)")
fig.update_layout(template="plotly_dark", height=320)
fig.show()

# Forward pass
dec_layer = DecoderLayer(d_model=32, num_heads=4, d_ff=64, rng=np.random.default_rng(7))
tgt = np.random.default_rng(5).normal(size=(5, 32))
dec_out = dec_layer.forward(tgt, enc_out)
print(f"Decoder layer output: {dec_out.shape}  (tgt_len=5, d_model=32)")

### 2.6 Full Encoder-Decoder: Complexity & Parameter Count

#### Parameter Count

Let $N_e$ = encoder layers, $N_d$ = decoder layers, $d$ = model dimension, $d_{\text{ff}} = 4d$, $V$ = vocabulary size.

| Component | Parameters |
|---|---|
| Token embedding | $V \times d$ |
| Encoder (per layer) | $12\,d^2$ |
| Decoder (per layer) | $16\,d^2$ (extra cross-attention) |
| Output projection | $d \times V$ (often tied to embedding) |
| **Total** | $2Vd + 12 N_e d^2 + 16 N_d d^2$ |

For T5-base: $V=32{,}128$, $d=768$, $N_e = N_d = 12$, giving $\approx 220$M parameters.

#### Training Complexity (per batch step)

| Operation | FLOPs |
|---|---|
| Encoder self-attention | $O(N_e \cdot n^2 d + N_e \cdot n d^2)$ |
| Decoder self-attention (masked) | $O(N_d \cdot m^2 d + N_d \cdot m d^2)$ |
| Decoder cross-attention | $O(N_d \cdot m n d + N_d \cdot m d^2)$ |
| FFN (both stacks) | $O((N_e + N_d) \cdot (n+m) d^2)$ |

For $n, m \ll d$ (typical at scale), FFN dominates. For long sequences, attention is the bottleneck.

#### Inference: Beam Search

At inference, the decoder generates via **beam search** with beam width $B$:

$$Y^* = \arg\max_{Y} \log P(Y \mid X) / |Y|^\alpha$$

where $|Y|^\alpha$ is a length normalisation penalty (typically $\alpha \approx 0.6$).

Beam search maintains $B$ partial hypotheses, expanding each by all $|\mathcal{V}|$ tokens and keeping the top-$B$ by cumulative log-probability. Total cost: $O(T \cdot B \cdot |\mathcal{V}|)$ scoring steps.

In [ ]:
class EncoderDecoder:
    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 num_enc_layers: int, num_dec_layers: int, rng):
        self.encoder = [EncoderLayer(d_model, num_heads, d_ff, rng)
                        for _ in range(num_enc_layers)]
        self.decoder = [DecoderLayer(d_model, num_heads, d_ff, rng)
                        for _ in range(num_dec_layers)]
        self.pe = sinusoidal_pe(512, d_model)

    def encode(self, src: np.ndarray) -> np.ndarray:
        x = src + self.pe[: src.shape[0]]
        for layer in self.encoder:
            x = layer.forward(x)
        return x

    def decode(self, tgt: np.ndarray, memory: np.ndarray) -> np.ndarray:
        x = tgt + self.pe[: tgt.shape[0]]
        for layer in self.decoder:
            x = layer.forward(x, memory)
        return x

    def forward(self, src: np.ndarray, tgt: np.ndarray) -> np.ndarray:
        memory = self.encode(src)
        return self.decode(tgt, memory)


model = EncoderDecoder(
    d_model=32, num_heads=4, d_ff=64,
    num_enc_layers=2, num_dec_layers=2,
    rng=np.random.default_rng(0)
)
src = np.random.default_rng(1).normal(size=(10, 32))  # 10-token source
tgt = np.random.default_rng(2).normal(size=(7, 32))   # 7-token target
out = model.forward(src, tgt)
print(f"Encoder-Decoder output: {out.shape}  (tgt_len=7, d_model=32)")
print("\nParameter counts (approximate):")
print(f"  Encoder layers : 2")
print(f"  Decoder layers : 2")
print(f"  d_model        : 32")
print(f"  Heads          : 4")

In [ ]:
# Verify parameter count formula against the actual model object
def count_params_formula(d: int, d_ff: int, n_enc: int, n_dec: int, vocab: int) -> dict:
    per_enc = 4 * d**2 + 2 * d * d_ff          # MHA + FFN (ignoring biases/norms)
    per_dec = 4 * d**2 + 4 * d**2 + 2 * d * d_ff  # self-attn + cross-attn + FFN
    embed   = vocab * d
    total   = embed + n_enc * per_enc + n_dec * per_dec
    return {"embed": embed, "per_encoder_layer": per_enc, "per_decoder_layer": per_dec,
            "total_approx": total}


configs = {
    "Tiny (demo)":    dict(d=32,  d_ff=128,  n_enc=2,  n_dec=2,  vocab=256),
    "T5-small":       dict(d=512, d_ff=2048, n_enc=6,  n_dec=6,  vocab=32128),
    "T5-base":        dict(d=768, d_ff=3072, n_enc=12, n_dec=12, vocab=32128),
    "T5-large":       dict(d=1024,d_ff=4096, n_enc=24, n_dec=24, vocab=32128),
}

print(f"{'Config':20s}  {'Embed':>12}  {'Enc layer':>12}  {'Dec layer':>12}  {'Total':>14}")
print("-" * 80)
for name, cfg in configs.items():
    p = count_params_formula(**cfg)
    print(f"{name:20s}  {p['embed']:12,}  {p['per_encoder_layer']:12,}  "
          f"{p['per_decoder_layer']:12,}  {p['total_approx']:14,}")

# Plot scaling
total_params = [count_params_formula(**cfg)["total_approx"] for cfg in configs.values()]
fig = go.Figure(go.Bar(
    x=list(configs.keys()), y=total_params,
    text=[f"{p/1e6:.0f}M" for p in total_params], textposition="outside",
    marker_color=px.colors.qualitative.Plotly[:4],
))
fig.update_layout(title="Parameter Count by Model Size",
                  yaxis_title="Parameters", yaxis_type="log",
                  template="plotly_dark", height=380)
fig.show()

---
## 3 — Practical: Encoder-Decoder with HuggingFace (T5)

T5 (Text-To-Text Transfer Transformer) frames every NLP task as text-to-text,  
using a Unigram (SentencePiece) tokenizer and a standard encoder-decoder Transformer.

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.eval()

def translate(text: str, prefix: str = "translate English to French: ") -> str:
    inputs = tokenizer(prefix + text, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=60)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is a critical step in natural language processing.",
    "Encoder-decoder models are used for sequence-to-sequence tasks.",
]
for s in sentences:
    print(f"EN: {s}")
    print(f"FR: {translate(s)}\n")

In [ ]:
# Inspect encoder output and cross-attention weights
text = "The encoder builds contextual representations."
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    enc_outputs = model.encoder(**inputs, output_attentions=True)

# Last encoder layer, first attention head
attn = enc_outputs.attentions[-1][0, 0].numpy()  # (seq, seq)

fig = px.imshow(
    attn,
    x=tokens, y=tokens,
    color_continuous_scale="Blues",
    title="T5 Encoder — Last Layer, Head 0 — Self-Attention",
    labels=dict(color="Weight"),
)
fig.update_layout(template="plotly_dark", height=450)
fig.show()

---
## Summary

### Tokenisation Algorithms

| | BPE | WordPiece | Unigram LM |
|---|---|---|---|
| **Direction** | bottom-up | bottom-up | top-down |
| **Objective** | max frequency (MDL proxy) | max corpus log-likelihood | max corpus log-likelihood |
| **Merge criterion** | $\arg\max\,\text{freq}(ab)$ | $\arg\max\,\frac{\text{freq}(ab)}{\text{freq}(a)\,\text{freq}(b)}$ | $\arg\min\,\Delta\mathcal{L}(t)$ |
| **Segmentation** | deterministic | deterministic | probabilistic (Viterbi / sample) |
| **OOV** | always decomposes to chars | always decomposes to chars | always decomposes to chars |
| **Training** | $O(KN)$ | $O(KN)$ | $O(\text{EM iters} \times N \ell^2)$ |
| **Used by** | GPT-*, LLaMA, Mistral | BERT, DistilBERT, ELECTRA | T5, mBART, LLaMA (SP) |

### Encoder-Decoder Mathematics

| Component | Key equation | Complexity |
|---|---|---|
| **Attention** | $\text{softmax}(QK^\top/\sqrt{d_k})V$ | $O(n^2 d)$ time, $O(n^2)$ space |
| **Scaling** | $\text{Var}[q \cdot k] = d_k$ without scaling | $O(1)$ fix: divide by $\sqrt{d_k}$ |
| **MHA** | $h$ independent heads, $4d^2$ total params | independent of $h$ |
| **Sinusoidal PE** | $PE_{\text{pos}+k} = R_k \cdot PE_{\text{pos}}$ | $O(nd)$ |
| **RoPE** | $\langle f(q,m), f(k,n)\rangle = g(q,k,m{-}n)$ | $O(nd)$ |
| **LayerNorm** | normalise over $d$, learnable $\gamma, \beta$ | $O(nd)$ |
| **FFN** | $W_2\,\sigma(W_1 x)$, $d_{\text{ff}}=4d$ | $O(nd^2)$ |
| **Full model** | $2Vd + 12N_e d^2 + 16N_d d^2$ params | dominates at scale |
| **KV cache** | $2 N_\ell h d_k T B$ bytes | linear in sequence length |
| **Beam search** | $\arg\max \log P(Y\mid X)/|Y|^\alpha$ | $O(T B |\mathcal{V}|)$ |

### Design Decisions in Modern LLMs

| Choice | Original Transformer | Modern LLMs (LLaMA 3, Mistral) |
|---|---|---|
| Normalisation | Post-LN | Pre-LN / RMSNorm |
| Activation | ReLU | SwiGLU |
| Positional encoding | Sinusoidal | RoPE |
| Attention heads | MHA | GQA |
| Architecture | Encoder-Decoder | Decoder-Only |